# Customer Churn Preprocessing Pipeline

This notebook performs the complete preprocessing pipeline for the Telco Customer Churn dataset. It cleans data, encodes variables, scales features, splits into train/test sets, applies SMOTE, and saves all processed files.

## STEP 1 — Load & Clean

Load the CSV, convert `TotalCharges` to float, impute 11 missing values with `0.0`, and drop `customerID` (high cardinality, no predictive value).

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

# Resolve base directory regardless of where notebook is run from
BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Load dataset
data_path = os.path.join(BASE_DIR, 'data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(data_path)
print(f"Initial shape: {df.shape}")

# Convert TotalCharges to float, coerce errors to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Impute 11 missing TotalCharges with 0.0 (new customers, tenure=0)
print(f"Missing TotalCharges before imputation: {df['TotalCharges'].isnull().sum()}")
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)
print(f"Missing TotalCharges after imputation: {df['TotalCharges'].isnull().sum()}")

# Drop customerID (unique identifier, no predictive value)
df = df.drop(columns=['customerID'])
print(f"Shape after dropping customerID: {df.shape}")
df.head()

## STEP 2 — Encode Target Variable

Map the binary target `Churn`: `Yes` → `1`, `No` → `0`.

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print("Target class distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {df['Churn'].mean()*100:.2f}%")

## STEP 3 — Encode Categorical Features

- **Ordinal Encoding** on `Contract` (preserves rank: Month-to-month=0, One year=1, Two year=2)
- **One-Hot Encoding** (`drop_first=True`) on all remaining nominal columns to avoid multicollinearity.

In [ ]:
# Ordinal encode Contract
contract_mapping = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df['Contract'] = df['Contract'].map(contract_mapping)

# Nominal columns for One-Hot Encoding
nominal_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling',
    'PaymentMethod'
]

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True, dtype=int)
print(f"Shape after encoding: {df.shape}")
print("Columns:", df.columns.tolist())

## STEP 4 — Feature Scaling

Apply `StandardScaler` to `tenure`, `MonthlyCharges`, and `TotalCharges` only. Binary/encoded columns are left unchanged.

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print("Scaled numerical columns (first 3 rows):")
print(df[numerical_cols].head(3))

## STEP 5 — Train-Test Split

Split 80/20 with `stratify=y` to preserve the class balance across both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")
print("\nClass distribution in y_train:")
print(y_train.value_counts(normalize=True).round(4))
print("\nClass distribution in y_test:")
print(y_test.value_counts(normalize=True).round(4))

## STEP 6 — Handle Class Imbalance with SMOTE

SMOTE is applied **only** to the training set to prevent data leakage. The test set remains untouched.

In [ ]:
from imblearn.over_sampling import SMOTE

print("Class distribution before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nClass distribution after SMOTE:")
import pandas as pd
print(pd.Series(y_train_res).value_counts())
print(f"\nX_train shape after SMOTE: {X_train_res.shape}")

## STEP 7 — Save Processed Data

Save all four arrays to `processed/` and the fitted scaler to `models/` for use in training and inference.

In [ ]:
# Save processed splits to processed/ folder
joblib.dump(X_train_res, os.path.join(PROCESSED_DIR, 'X_train.pkl'))
joblib.dump(X_test,      os.path.join(PROCESSED_DIR, 'X_test.pkl'))
joblib.dump(y_train_res, os.path.join(PROCESSED_DIR, 'y_train.pkl'))
joblib.dump(y_test,      os.path.join(PROCESSED_DIR, 'y_test.pkl'))
joblib.dump(scaler,      os.path.join(MODELS_DIR,    'scaler.pkl'))

print(f"Saved → {os.path.join(PROCESSED_DIR, 'X_train.pkl')}")
print(f"Saved → {os.path.join(PROCESSED_DIR, 'X_test.pkl')}")
print(f"Saved → {os.path.join(PROCESSED_DIR, 'y_train.pkl')}")
print(f"Saved → {os.path.join(PROCESSED_DIR, 'y_test.pkl')}")
print(f"Saved → {os.path.join(MODELS_DIR, 'scaler.pkl')}")
print("\nAll preprocessing files saved successfully.")